In [ ]:
# Setup: in Google Colab, download and install the repository (a few minutes, once per session); elsewhere, work from the
# repository. Pages that read the derived catalogues need Zenodo bundle 1 (python scripts/fetch_release.py).
import os, subprocess, sys
os.environ['MPLBACKEND'] = 'Agg'   # as in a plain script run: Jupyter's inline settings would change the saved figures
if 'google.colab' in sys.modules:
    if not os.path.exists('/content/lmc-ancient-disc'):
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/vasilybelokurov/lmc-ancient-disc', '/content/lmc-ancient-disc'], check=True)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '/content/lmc-ancient-disc'], check=True)
    os.chdir('/content/lmc-ancient-disc')
else:
    while not os.path.exists('pyproject.toml'):
        os.chdir('..')
sys.path[:0] = ['src', 'scripts', 'kb']
if not os.path.exists('data/giants_apogee.parquet'):   # the catalogues of Zenodo bundle 1, checked against data/sources.yaml
    subprocess.run([sys.executable, 'scripts/fetch_release.py'], check=True)

# Data: RR Lyrae and BOSS calibration

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/lmc-ancient-disc/blob/main/kb/notebooks/chapters/data_calibration.ipynb)

Every number on this page is written by `scripts/paper_text_data.py` from the result tables and catalogues
(`results/macros_data.tex`); the code cells are the figure scripts `scripts/paper_rrab_membership.py`, `scripts/paper_rrab_feh.py`,
`scripts/paper_boss_calibration.py` and `scripts/paper_boss_hot_bias.py`, step by step, and the two RR Lyrae diagnostics behind
them, shown but not run.

This chapter supports the samples chapter: how the RR Lyrae members are chosen, how their two photometric metallicity scales are
joined, how the BOSS spectra of the red giants are placed on the APOGEE scale, and where that calibration fails. RRab are RR Lyrae
stars pulsating in the fundamental mode; [Fe/H] is the iron abundance relative to the Sun's (dex); $\rho$ is the angular distance
from the LMC centre and $\Delta\mu$ the distance modulus of a star relative to the LMC centre (mag), both defined in the samples
chapter.

## RR Lyrae membership

Figure 3.1 shows how the RR Lyrae members are selected. The SMC mask removes stars within
13° of the centre of the Small Magellanic Cloud (SMC), the radius beyond which the fraction of RRab with
SMC-like proper motions (closer to the SMC's systemic motion than to the LMC's) falls to the level set by LMC stars. The distance
window $|\Delta\mu|$ < 0.6 mag removes the foreground and background; the density in the sidebands
0.9 < $|\Delta\mu|$ < 2 mag, assumed flat in $\Delta\mu$, gives the foreground left in the
window, at most 2.3 per cent inside 10° and 6 per cent at
10–12°.

<details><summary>About <code>scripts/paper_rrab_membership.py</code></summary>

```text
Paper figure: RR Lyrae membership. (a) sky to 20 deg: members, stars in the SMC mask, stars outside the distance window;
(b) relative distance modulus against distance from the LMC centre; (c) foreground left in the distance window against radius,
with, inset, the fraction of stars with SMC-like proper motion against distance from the SMC, which sets the mask radius.
Reads data/rrab.parquet and results/rrab_membership.csv; writes figures/paper_rrab_membership.png.

Inputs: data/rrab.parquet (sources.yaml release_rrab, scripts/build_rrab.py), RRab with a relative distance modulus dmu (stars
without one are dropped here), smc_masked flag, rho [deg] and Gaia proper motions; results/rrab_membership.csv
(scripts/fig_rrab_membership.py): per radial bin rho_lo-rho_hi [deg] the foreground fraction left in the distance window, contam,
and its error contam_err, from the density of stars in the sidebands 0.9 < |dmu| < 2 mag assumed flat in dmu.
Limits from lmcold.footprint: DMU_WINDOW = 0.6 mag, RHO_MAX = 10 deg, SMC_MASK_RADIUS = 13 deg.
Panels (1 x 3, square):
(a) x (east, increasing to the left), y (north) [deg], orthographic about the LMC centre, +-25 deg: members (blue, outside the
    SMC mask and |dmu| < 0.6), SMC-masked stars (orange), stars outside the window (yellow); circles: SMC mask (orange),
    10-deg science radius (dashed), 20-deg limit of the data (grey).
(b) x: rho [deg], 0-20; y: dmu [mag], -2 to 2 (axis to 2.4 to leave room for the colour bar); 2D histogram (0.4 deg x 0.05 mag,
    log colour) of stars outside the SMC mask; orange lines: the +-0.6 mag window; dashed: RHO_MAX.
(c) x: rho [deg]; y: foreground in the window [per cent] (log, 0.4-150), x error bars the bin half-widths. Inset: fraction of
    window stars whose proper motion is closer to the SMC systemic motion than to the LMC one, in 1-deg bins of distance from
    the SMC centre [deg]; orange line: mask radius.
Placement: appendix (Sect. 2.1; docs/figures.md). Run: .venv/bin/python scripts/paper_rrab_membership.py   (about 1 s on a laptop, docs/run_times.txt)
```
</details>

**Data read by `scripts/paper_rrab_membership.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `data/rrab.parquet` | Zenodo bundle 1 | RRab within 20 deg: OGLE-IV + Gaia DR3, [Fe/H], dmu, membership |
| `results/rrab_membership.csv` | repository | RR Lyrae foreground fraction per ring; written by `python scripts/fig_rrab_membership.py` |

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LogNorm

from lmcold.constants import LMC, SMC
from lmcold.footprint import DMU_WINDOW, RHO_MAX, SMC_MASK_RADIUS, angular_distance, lmc_xy, small_circle
from lmcold.layout import panel_grid
from lmcold.sources import DATA, ROOT

plt.style.use(ROOT / "src" / "lmcold" / "paper.mplstyle")
# Colours: members, SMC mask, stars outside the distance window, 20-deg data limit.
BLUE, ORANGE, YELLOW, GREY = "#2a78d6", "#eb6834", "#eda100", "0.35"

### Load the data
The RRab catalogue `data/rrab.parquet` (Zenodo bundle 1) and the foreground fraction per radial bin from
`results/rrab_membership.csv` (written by `scripts/fig_rrab_membership.py`).

In [ ]:
# data/rrab.parquet: RRab with relative distance modulus dmu, smc_masked flag, rho [deg] and Gaia proper motions
t = pd.read_parquet(DATA / "rrab.parquet")
# results/rrab_membership.csv: per radial bin (rho_lo, rho_hi) the foreground fraction left in the window, contam, contam_err
c = pd.read_csv(ROOT / "results" / "rrab_membership.csv")

### The stars with a distance
RRab with a relative distance modulus, in the orthographic projection, and those inside the distance window.

In [ ]:
t = t[t.dmu.notna()]
# Orthographic x (east), y (north) [deg] about the LMC centre.
t["x"], t["y"] = lmc_xy(t.ra.to_numpy(), t.dec.to_numpy())
win = t.dmu.abs() < DMU_WINDOW  # inside the distance window

### Panel (a): the sky
Members, SMC-masked stars and stars outside the distance window.

In [ ]:
# One row of three equal panels.
fig, (ax,) = panel_grid(1, 3, top=0.1)

a = ax[0]
# Three groups, members drawn last (on top) with smaller points.
for sel, col, lab, s in ((~t.smc_masked & ~win, YELLOW, r"$|\Delta\mu| > 0.6$", 0.6), (t.smc_masked, ORANGE, "SMC", 0.6),
                         (~t.smc_masked & win, BLUE, r"$|\Delta\mu| < 0.6$", 0.15)):
    a.scatter(t.x[sel], t.y[sel], s=s, color=col, lw=0, rasterized=True, label=lab)
# Circles: the SMC mask, the 10-deg science radius RHO_MAX and the 20-deg data limit, projected about the LMC centre.
for (ra0, dec0, r), col, ls in (((SMC["ra0"], SMC["dec0"], SMC_MASK_RADIUS), ORANGE, "-"), ((LMC["ra0"], LMC["dec0"], RHO_MAX), "k", "--"),
                                ((LMC["ra0"], LMC["dec0"], 20.0), GREY, "-")):
    cx, cy = lmc_xy(*small_circle(ra0, dec0, r))
    front = np.hypot(cx, cy) <= 20.01  # draw only the part of each circle within the 20 deg data circle
    a.plot(np.where(front, cx, np.nan), np.where(front, cy, np.nan), color=col, ls=ls, lw=0.8)
a.set(xlim=(25, -25), ylim=(-25, 25), aspect="equal", xlabel=r"$x$ (east) [deg]", ylabel=r"$y$ (north) [deg]")
a.legend(loc="lower left", ncol=3, fontsize=6, markerscale=6, handletextpad=0.05, borderaxespad=0.1, columnspacing=0.6,
         handlelength=0.8)  # one row in the strip below y = -21, outside the 20 deg data circle
a.text(0.96, 0.04, "(a)", transform=a.transAxes, ha="right")

### Panel (b): relative distance modulus against $\rho$

In [ ]:
a = ax[1]
# 2D histogram of the stars outside the SMC mask: 0.4-deg bins in rho, 0.05-mag bins in dmu, log colour scale.
o = ~t.smc_masked
h = a.hist2d(t.rho[o], t.dmu[o], bins=[np.arange(0, 20.01, 0.4), np.arange(-2.0, 2.01, 0.05)], norm=LogNorm(), cmap="Blues",
             rasterized=True)
# The +-DMU_WINDOW distance window (orange) and RHO_MAX (dashed).
for s in (1, -1):
    a.axhline(s * DMU_WINDOW, color=ORANGE, lw=0.8)
a.plot([RHO_MAX, RHO_MAX], [-2.0, 2.0], color="k", ls="--", lw=0.8)
a.set(xlabel=r"$\rho$ [deg]", ylabel=r"$\Delta\mu$ [mag]")
a.set_ylim(-2.0, 2.4)  # an empty strip above the data (|dmu| <= 2) holds the colour bar
# Horizontal colour bar inside the empty strip at the top of the panel.
cax = a.inset_axes([0.6, 0.855, 0.36, 0.045])
fig.colorbar(h[3], cax=cax, orientation="horizontal")
cax.tick_params(labelsize=5, pad=1, length=2)
a.text(0.78, 0.94, "stars/pixel", transform=a.transAxes, ha="center", va="center", fontsize=6)
a.text(0.96, 0.04, "(b)", transform=a.transAxes, ha="right")

### Panel (c): foreground in the window
With, inset, the fraction of stars with SMC-like proper motion against distance from the SMC.

In [ ]:
a = ax[2]
# Bin centres [deg]; foreground fraction in per cent, x error bars the bin half-widths.
rc = 0.5 * (c.rho_lo + c.rho_hi)
a.errorbar(rc, 100 * c.contam, yerr=100 * c.contam_err, xerr=0.5 * (c.rho_hi - c.rho_lo), fmt="o", ms=2.5, color=BLUE, lw=0.8)
a.axvline(RHO_MAX, color="k", ls="--", lw=0.8)
a.set(yscale="log", ylim=(0.4, 150), xlabel=r"$\rho$ [deg]", ylabel=r"foreground in the window [%]")
a.text(0.96, 0.04, "(c)", transform=a.transAxes, ha="right")
# Inset: stars in the distance window with a Gaia proper motion.
ins = a.inset_axes([0.17, 0.66, 0.27, 0.30])
d = t[win & t.pmra.notna()]
# SMC-like: proper motion [mas/yr] nearer to the SMC systemic motion than to the LMC one (lmcold.constants).
smclike = np.hypot(d.pmra - SMC["pmra0"], d.pmdec - SMC["pmdec0"]) < np.hypot(d.pmra - LMC["pmra0"], d.pmdec - LMC["pmdec0"])
# Distance from the SMC centre [deg] in 1-deg bins, 0-22 deg; the SMC-like fraction per bin, and the mask radius (orange line).
dS = angular_distance(d.ra, d.dec, SMC["ra0"], SMC["dec0"])
b = np.arange(0, 22.1, 1.0)
i = np.digitize(dS, b)
ins.plot(0.5 * (b[1:] + b[:-1]), [smclike[i == k].mean() for k in range(1, len(b))], "o-", ms=1.5, lw=0.8, color=BLUE)
ins.axvline(SMC_MASK_RADIUS, color=ORANGE, lw=0.8)
ins.set_xlabel("from SMC [deg]", fontsize=6, labelpad=1)
ins.set_ylabel("SMC-like", fontsize=6, labelpad=2)
ins.tick_params(labelsize=6)
ins.set(ylim=(0, 1.05), xticks=[0, 10, 20], yticks=[0, 0.5, 1])

fig.savefig(ROOT / "figures" / "paper_rrab_membership.png")

In [ ]:
from IPython.display import Image
Image(filename='figures/paper_rrab_membership.png')

**Figure 3.1.** RR Lyrae membership. (a) The sky to 20°, in the orthographic projection
about the LMC centre, $x$ (east, increasing to the left) and $y$ (north) in degrees: members (blue, $|\Delta\mu|$ <
0.6 mag outside the SMC mask), stars in the SMC mask (orange) and stars outside the distance window (yellow);
orange circle: the SMC mask; dashed circle: the 10° science radius; grey circle: the
20° limit of the data. (b) Relative distance modulus $\Delta\mu$ (mag) against $\rho$ (deg) for the stars
outside the SMC mask (two-dimensional histogram, logarithmic colour scale in stars per pixel); orange lines: the window
$|\Delta\mu|$ < 0.6 mag; dashed line: the science radius. (c) Foreground left in the window (per cent,
logarithmic axis) against $\rho$, from the $\Delta\mu$ sidebands, with horizontal bars the widths of the radial bins; inset:
fraction of window stars with SMC-like proper motions against distance from the SMC centre (deg), with the mask radius (orange
line). `scripts/paper_rrab_membership.py`

### The membership diagnostic

`scripts/fig_rrab_membership.py` is the diagnostic behind the membership limits. Per radial bin it counts the stars removed by the
SMC mask, the members and the sideband stars, and scales the sideband counts to the width of the window to estimate the
foreground left among the members. It writes these numbers to `results/rrab_membership.csv`, which panel (c) of
Figure 3.1 and the text use, and draws a figure of its own that is not part of the paper.

<details><summary>About <code>scripts/fig_rrab_membership.py</code></summary>

```text
RRab membership: the SMC mask, the distance window, and what is left behind.

(a) sky: members, stars removed by the SMC mask, stars removed by the distance window; (b) fraction of window stars with SMC-like
proper motion against distance from the SMC (closer to the SMC systemic PM than to the LMC one), which sets the mask radius;
(c) dmu in radial bins outside the mask, with the window and the sidebands; (d) residual contamination inside the window,
estimated from the sideband density (assumed flat in dmu); (e) counts against radius at each step.
Writes figures/rrab_membership.png and results/rrab_membership.csv.

Diagnostic behind the membership limits of lmcold.footprint (SMC mask radius 13 deg, |dmu| < 0.6 mag, rho < 10 deg); the paper
version is scripts/paper_rrab_membership.py (appendix, docs/figures.md), which reads results/rrab_membership.csv for its panel (c);
scripts/paper_text_data.py takes the foreground fractions of the text from the same table.

Input: data/rrab.parquet (sources.yaml key release_rrab): ra, dec, rho [deg], dmu [mag, relative distance modulus], pmra, pmdec
[mas/yr], smc_masked, member. Stars without dmu are dropped. Reference proper motions: lmcold.constants.LMC and SMC.

Figure (2 x 3 grid, matplotlib constrained layout):
(a) (left column, both rows) sky in orthographic coordinates about the LMC centre (lmcold.footprint.lmc_xy), x east [deg]
    (increasing to the left), y north [deg]: members (blue), stars in the SMC mask (orange), stars outside it but outside the
    distance window (yellow); orange curve: the mask edge; grey circle: radius 20 in these coordinates (the 20-deg catalogue
    edge lies at 19.6).
(b) x: angular distance from the SMC centre [deg], 1-deg bins; y: fraction of window stars with Gaia proper motions closer to the
    SMC systemic proper motion than to the LMC one, with binomial errors; vertical line: mask radius.
(c) x: dmu [mag]; y (log): normalised histograms outside the SMC mask in four rho ranges [deg]; grey band: the window, light
    orange bands: the sidebands SIDE.
(d) x: rho [deg] (bins RB); y (log): residual contamination of the window [per cent], with Poisson errors from the
    sideband counts.
(e) x: rho [deg]; y (log): RRab per bin with dmu (grey), removed by the SMC mask (orange), members (blue; window and outside the
    mask, before the rho < 10 deg cut).

results/rrab_membership.csv, per rho bin: rho_lo, rho_hi [deg], N (with dmu), masked, members (window, outside the mask), side
(sideband stars outside the mask), contam and contam_err (expected foreground in the window / members).

Run: .venv/bin/python scripts/fig_rrab_membership.py
```
</details>

*Shown, not run here: data/rrab.parquet (scripts/build_rrab.py); a diagnostic whose figure is not on this site. The cells below are the script itself; run it from the repository with* `python scripts/fig_rrab_membership.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

```python
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from lmcold.constants import LMC, SMC
from lmcold.footprint import DMU_WINDOW, SMC_MASK_RADIUS, angular_distance
from lmcold.footprint import lmc_xy
from lmcold.sources import DATA, ROOT
```

#### Settings
The $\Delta\mu$ sidebands and the radial bins.

```python
SIDE = (0.9, 2.0)  # sideband |dmu| range [mag], beyond the LMC distribution
RB = np.array([0, 2, 4, 6, 8, 10, 12, 14, 17, 20])  # rho bins [deg]


xy = lmc_xy  # (ra, dec) [deg] -> orthographic (x, y) [deg] about the LMC centre
```

#### Load the data
The RR Lyrae catalogue `data/rrab.parquet` (`scripts/build_rrab.py`).

```python
# one row per RRab; columns listed in the module docstring
t = pd.read_parquet(DATA / "rrab.parquet")
```

#### Window and sidebands
Stars with a distance, their projected position and distance from the SMC, and the
window and sideband flags.

```python
# drop stars without a relative distance modulus
t = t[t.dmu.notna()]
t["x"], t["y"] = xy(t.ra, t.dec)  # orthographic coordinates [deg]
t["dS"] = angular_distance(t.ra, t.dec, SMC["ra0"], SMC["dec0"])  # angular distance from the SMC centre [deg]
win = t.dmu.abs() < DMU_WINDOW  # distance window, |dmu| < DMU_WINDOW = 0.6 mag (lmcold.footprint)
side = (t.dmu.abs() > SIDE[0]) & (t.dmu.abs() < SIDE[1])  # sidebands, 0.9 < |dmu| < 2.0 mag
```

#### Foreground in the window
Per radial bin, the sideband counts scaled to the window width give the foreground left among
the members; written to `results/rrab_membership.csv`.

```python
rows = []
for lo, hi in zip(RB[:-1], RB[1:]):
    # stars in this rho bin, and those of them outside the SMC mask
    r = (t.rho >= lo) & (t.rho < hi)
    ok = r & ~t.smc_masked
    n_side = (ok & side).sum()
    # Sideband counts scaled by width: window 2 x 0.6 = 1.2 mag against sidebands 2 x (2.0 - 0.9) = 2.2 mag.
    expect = n_side * (2 * DMU_WINDOW) / (2 * (SIDE[1] - SIDE[0]))  # foreground expected inside the window
    # contam: expected foreground / members; contam_err: Poisson error of the sideband count propagated to contam
    # (max(..., 1) avoids division by zero in empty bins)
    rows.append(dict(rho_lo=lo, rho_hi=hi, N=r.sum(), masked=(r & t.smc_masked).sum(), members=(ok & win).sum(),
                     side=n_side, contam=expect / max((ok & win).sum(), 1), contam_err=np.sqrt(max(n_side, 1)) * expect /
                     max(n_side, 1) / max((ok & win).sum(), 1)))
res = pd.DataFrame(rows)
res.to_csv(ROOT / "results" / "rrab_membership.csv", index=False, float_format="%.4f")
print(res.to_string(index=False, float_format="%.3f"))
```

#### Panel (a): the sky
Members, stars in the SMC mask and stars outside the distance window.

```python
# 2 x 3 grid: panel (a) fills the left column
fig = plt.figure(figsize=(17, 9.5), constrained_layout=True)
g = fig.add_gridspec(2, 3)
a = fig.add_subplot(g[:, 0])
# per class: selection, colour, label, marker size
for sel, c, lab, s in ((~t.smc_masked & ~win, "#eda100", "outside the distance window", 2),
                       (t.smc_masked, "#eb6834", "SMC mask", 2), (t.member, "#2a78d6", "members", 0.5)):
    a.scatter(t.x[sel], t.y[sel], s=s, color=c, label=f"{lab} ({sel.sum()})", rasterized=True)
# grey circle of radius 20 deg in the orthographic coordinates
th = np.linspace(0, 2 * np.pi, 400)
a.plot(20 * np.sin(th), 20 * np.cos(th), color="0.3", lw=0.8)
ra_m, dec_m = [], []
# Same small-circle formula as lmcold.footprint.small_circle.
for p in th:  # mask edge: points 13 deg from the SMC centre
    # d: mask radius; r0, a0: SMC centre dec and ra [rad]; p: position angle on the circle [rad]
    d, r0, a0 = np.deg2rad(SMC_MASK_RADIUS), np.deg2rad(SMC["dec0"]), np.deg2rad(SMC["ra0"])
    dec = np.arcsin(np.sin(r0) * np.cos(d) + np.cos(r0) * np.sin(d) * np.cos(p))
    ra_m.append(np.rad2deg(a0 + np.arctan2(np.sin(p) * np.sin(d) * np.cos(r0), np.cos(d) - np.sin(r0) * np.sin(dec))))
    dec_m.append(np.rad2deg(dec))
# mask edge in the orthographic coordinates
mx, my = xy(np.array(ra_m), np.array(dec_m))
a.plot(mx, my, color="#eb6834", lw=1.5)
# x axis reversed: east to the left
a.set(xlim=(21, -21), ylim=(-21, 21), aspect="equal", xlabel="x east [deg]", ylabel="y north [deg]", title="(a) membership")
a.legend(frameon=False, fontsize=8, loc="upper right", markerscale=4)
```

#### Panel (b): the SMC mask radius
Fraction of window stars with SMC-like proper motions against distance
from the SMC centre.

```python
a = fig.add_subplot(g[0, 1])
# SMC-like: proper motion closer (Euclidean, in mas/yr) to the SMC systemic proper motion than to the LMC one.
smclike = np.hypot(t.pmra - SMC["pmra0"], t.pmdec - SMC["pmdec0"]) < np.hypot(t.pmra - LMC["pmra0"], t.pmdec - LMC["pmdec0"])
# 1-deg bins of distance from the SMC centre [deg]
b = np.arange(0, 22.1, 1.0)
# bin index of each window star with a Gaia proper motion
i = np.digitize(t.dS[win & t.pmra.notna()], b)
# f: fraction SMC-like per bin; n: stars per bin
f = [smclike[win & t.pmra.notna()][i == k].mean() for k in range(1, len(b))]
n = np.array([(i == k).sum() for k in range(1, len(b))])
# binomial error sqrt(f (1 - f) / n)
a.errorbar(0.5 * (b[1:] + b[:-1]), f, yerr=np.sqrt(np.array(f) * (1 - np.array(f)) / np.maximum(n, 1)), fmt="o-", color="#2a78d6")
a.axvline(SMC_MASK_RADIUS, color="#eb6834", lw=1.5, label=f"mask radius {SMC_MASK_RADIUS:g} deg")
a.set(xlabel="distance from the SMC centre [deg]", ylabel="fraction with SMC-like PM", title="(b) setting the mask radius", ylim=(0, 1.05))
a.legend(frameon=False)
```

#### Panel (c): relative distance modulus
Histograms of $\Delta\mu$ outside the SMC mask in four ranges of
$\rho$, with the window and the sidebands.

```python
a = fig.add_subplot(g[0, 2])
# dmu bins [mag], 0.05 mag wide
h = np.arange(-2.2, 2.21, 0.05)
# four rho ranges [deg], each with its colour
for (lo, hi), c in zip(((0, 6), (6, 10), (10, 14), (14, 20)), ("#2a78d6", "#1baf7a", "#eda100", "#eb6834")):
    sel = ~t.smc_masked & (t.rho >= lo) & (t.rho < hi)
    a.hist(t.dmu[sel], h, histtype="step", lw=1.8, color=c, density=True, label=f"rho {lo}-{hi} ({sel.sum()})")
a.axvspan(-DMU_WINDOW, DMU_WINDOW, color="0.85", zorder=0, label="window")
# sidebands on both sides of the window
for s in (1, -1):
    a.axvspan(s * SIDE[0], s * SIDE[1], color="#fbe3c4", zorder=0)
a.set(yscale="log", ylim=(1e-3, 5), xlabel="dmu [mag]", ylabel="density", title="(c) dmu outside the SMC mask (sidebands shaded)")
a.legend(frameon=False, fontsize=8)
```

#### Panels (d, e): foreground and counts
The foreground left in the window, and the counts at each step, against $\rho$.

```python
# centres of the rho bins [deg]
rc = 0.5 * (RB[1:] + RB[:-1])
a = fig.add_subplot(g[1, 1])
# contamination in per cent, with its Poisson error
a.errorbar(rc, 100 * res.contam, yerr=100 * res.contam_err, fmt="o-", color="#2a78d6")
a.set(xlabel="rho [deg]", ylabel="residual contamination in the window [%]", yscale="log", title="(d) contamination from sidebands")

a = fig.add_subplot(g[1, 2])
# counts per rho bin from the table above, drawn as steps
for col, c, lab in (("N", "0.4", "all with dmu"), ("masked", "#eb6834", "removed by the SMC mask"), ("members", "#2a78d6", "members")):
    a.step(RB, np.r_[res[col], res[col].iloc[-1]], where="post", color=c, lw=2, label=lab)
a.set(yscale="log", xlabel="rho [deg]", ylabel="RRab per bin", title="(e) counts")
a.legend(frameon=False, fontsize=8)
fig.savefig(ROOT / "figures" / "rrab_membership.png", dpi=150)
```

## RR Lyrae metallicities

Figure 3.2 compares the two photometric metallicity scales, both derived from the period and the light-curve phase
$\phi_{31}$ (samples chapter). The $G$-band values of [Muraveva et al. (2025)](https://doi.org/10.1093/mnras/stae2679) lie
0.22 dex below the $I$-band values of [Martínez-Vázquez et al. (2026)](https://doi.org/10.48550/arXiv.2609.27119)
with no trend with [Fe/H] (Deming slope 0.978 ± 0.012, which allows for errors in both), so a
constant shift brings them onto one scale. On the scale of [Crestani et al. (2021)](https://doi.org/10.3847/1538-4357/abd183), the
adopted values exceed the spectroscopic [Fe/H] of [Gratton et al. (2004)](https://doi.org/10.1051/0004-6361:20035840) by
0.07 ± 0.05 dex (48 stars) and those of
[Borissova et al. (2004](https://doi.org/10.1051/0004-6361:20034494), [2006)](https://doi.org/10.1051/0004-6361:20054132) by
0.14 ± 0.17 and 0.18 ± 0.07 dex (24 and
72 stars, from lower-resolution spectra). The RR Lyrae and giant [Fe/H] scales are therefore not identical; we do
not combine them in one metallicity axis.

<details><summary>About <code>scripts/paper_rrab_feh.py</code></summary>

```text
Paper figure: RRab photometric metallicities. (a) OGLE I-band against Gaia G-band (Muraveva+25) [Fe/H]; (b) their difference
against their mean, with the adopted constant shift and the Deming slope; (c) both against the spectroscopy of Gratton et al.
(2004), on the C21 scale, for the stars with all three. Reads data/rrab.parquet, results/rrab_feh_transform.csv and the
spectroscopic inputs (lmcold.spectra); writes figures/paper_rrab_feh.png.

Inputs: data/rrab.parquet (sources.yaml release_rrab): feh_I (OGLE I band, Crestani et al. 2021 scale) and feh_G_m25 (Gaia G
band, Muraveva et al. 2025 relation) [dex]; results/rrab_feh_transform.csv (scripts/fig_rrab_feh_transform.py): the median I - G
shift (column shift [dex]) and the Deming fit I = a + b G (deming_b, deming_b_err); the Gratton et al. (2004) spectroscopic
[Fe/H] (sources.yaml spec_gratton04 and the Di Fabrizio et al. 2005 positions, read by lmcold.spectra.read_lmc_rrl_spectra),
converted from the Harris (1996) ~ Zinn & West scale to the C21 scale with lmcold.rrlyrae.zw_to_c21 and matched to the RRab
within 2 arcsec.
Panels (1 x 3, square):
(a) x: [Fe/H]_G [dex], y: [Fe/H]_I [dex], both -3.4 to 0.2; 2D histogram (0.05 dex, log colour) of the stars with both; dotted
    one-to-one line; orange: the adopted constant shift.
(b) x: mean of the two [Fe/H] [dex], -3 to 0; y: I - G [dex], -1.5 to 1.9; 2D histogram; black: running median (solid) and
    16th/84th percentiles (dashed) in 0.15-dex bins with >= 30 stars; orange: the shift; the Deming slope is quoted above.
(c) x: spectroscopic [Fe/H] (C21 scale) [dex], -2.4 to -0.4; y: photometric [Fe/H] [dex]; blue: OGLE I, orange: Gaia G + shift;
    legend: median offset (photometric minus spectroscopic) and robust sd (1.4826 x MAD) [dex].
Placement: appendix (Sect. 2.1; docs/figures.md). Run: .venv/bin/python scripts/paper_rrab_feh.py   (about 2 s on a laptop, docs/run_times.txt)
```
</details>

**Data read by `scripts/paper_rrab_feh.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `data/raw/spec/J_AA_421_937/ReadMe` | public archive | CDS ReadMe (column layout) |
| `data/raw/spec/J_AA_421_937/table7.dat` | public archive | Gratton et al. 2004, A&A 421, 937 ([Fe/H], Harris 1996 scale) |
| `data/raw/spec/J_AA_423_97/ReadMe` | public archive | CDS ReadMe (column layout) |
| `data/raw/spec/J_AA_423_97/macho.dat` | public archive | Borissova et al. 2004, A&A 423, 97 (MACHO stars: [Fe/H], Harris 1996 scale) |
| `data/raw/spec/J_AA_430_603/table5.dat` | public archive | Di Fabrizio et al. 2005, A&A 430, 603: positions of field-A variables |
| `data/raw/spec/J_AA_430_603/table6.dat` | public archive | Di Fabrizio et al. 2005, A&A 430, 603: positions of field-B variables |
| `data/raw/spec/J_AA_460_459/ReadMe` | public archive | CDS ReadMe (column layout) |
| `data/raw/spec/J_AA_460_459/table9.dat` | public archive | Borissova et al. 2006, A&A 460, 459 ([Fe/H], Harris 1996 scale) |
| `data/raw/spec/simbad_macho.csv` | public archive | SIMBAD positions and main identifiers of the star names in Borissova et al. 2004, 2006 |
| `data/rrab.parquet` | Zenodo bundle 1 | RRab within 20 deg: OGLE-IV + Gaia DR3, [Fe/H], dmu, membership |
| `results/rrab_feh_transform.csv` | repository | RR Lyrae [Fe/H] scale transformation; written by `python scripts/fig_rrab_feh_transform.py` |

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
from matplotlib.colors import LogNorm

from lmcold.layout import panel_grid
from lmcold.rrlyrae import zw_to_c21
from lmcold.sources import DATA, ROOT
from lmcold.spectra import read_lmc_rrl_spectra

plt.style.use(ROOT / "src" / "lmcold" / "paper.mplstyle")
# colours of the OGLE I and Gaia G points
BLUE, ORANGE = "#2a78d6", "#eb6834"

### Load the data
The RRab catalogue `data/rrab.parquet` (Zenodo bundle 1), the I - G shift and Deming fit from `results/rrab_feh_transform.csv`
(written by `scripts/fig_rrab_feh_transform.py`), and the published RR Lyrae spectroscopy through
`lmcold.spectra.read_lmc_rrl_spectra()` (the VizieR tables under `data/raw/spec/`).

In [ ]:
# data/rrab.parquet: RRab with feh_I (OGLE I band) and feh_G_m25 (Gaia G band) [dex]
t = pd.read_parquet(DATA / "rrab.parquet")
# results/rrab_feh_transform.csv: median I - G shift [dex] and Deming slope deming_b, deming_b_err (one row)
tr = pd.read_csv(ROOT / "results" / "rrab_feh_transform.csv").iloc[0]
# lmcold.spectra.read_lmc_rrl_spectra(): spectroscopic [Fe/H] of LMC RR Lyrae with positions and source study (src)
g = read_lmc_rrl_spectra()

### The two photometric scales and the spectroscopy
RRab with both $I$- and $G$-band [Fe/H]; the Gratton et al. (2004) stars matched within 2 arcsec, on the
Crestani et al. (2021) scale.

In [ ]:
ov = t[t.feh_I.notna() & t.feh_G_m25.notna()]  # RRab with both photometric estimates
# mean and difference I - G of the two estimates [dex], (N,)
mean, diff = 0.5 * (ov.feh_I + ov.feh_G_m25), ov.feh_I - ov.feh_G_m25

g = g[g.src == "G04"]  # Gratton et al. (2004) only
# positions of the spectroscopic stars [deg]
cg = SkyCoord(g.ra, g.dec, unit="deg")
# Nearest RRab to each spectroscopic star; G is put on the I scale with the constant shift.
# j: index of the nearest RRab; sep: separation (angle)
j, sep, _ = cg.match_to_catalog_sky(SkyCoord(t.ra, t.dec, unit="deg"))
# spec: spectroscopic [Fe/H] converted to the C21 scale; I, G: photometric [Fe/H] of the matched RRab [dex]
s = pd.DataFrame(dict(spec=zw_to_c21(g.feh.to_numpy()), I=t.feh_I.to_numpy()[j], G=t.feh_G_m25.to_numpy()[j] + tr["shift"]))
# matches within 2 arcsec with all three [Fe/H] values
s = s[(sep.arcsec < 2) & s.I.notna() & s.G.notna() & s.spec.notna()]

### Panel (a): $I$-band against $G$-band [Fe/H]

In [ ]:
# one row of three square panels
fig, (ax,) = panel_grid(1, 3)
a = ax[0]
# 2D histogram of the stars with both estimates, 0.05-dex bins, log colour scale
h = a.hist2d(ov.feh_G_m25, ov.feh_I, bins=np.arange(-3.4, 0.21, 0.05), norm=LogNorm(), cmap="Blues", rasterized=True)
# dotted one-to-one line and the adopted shift (orange), over the axis range [dex]
x = np.array([-3.4, 0.2])
a.plot(x, x, color="0.4", lw=0.8, ls=":")
a.plot(x, x + tr["shift"], color=ORANGE, lw=1, label=f"$+{tr['shift']:.2f}$ dex")
a.set(xlabel=r"[Fe/H]$_G$ (Muraveva et al. 2025)", ylabel=r"[Fe/H]$_I$ (OGLE)", aspect="equal", xlim=(-3.4, 0.2), ylim=(-3.4, 0.2))
# legend above the panel, right-aligned
a.legend(loc="lower right", bbox_to_anchor=(1.0, 1.0), borderaxespad=0.1)
# panel letter and number of stars above the panel
a.text(0, 1.03, f"(a)  N = {len(ov)}", transform=a.transAxes)

### Panel (b): their difference against their mean
Running median and 16th/84th percentiles, the adopted shift and the Deming slope.

In [ ]:
a = ax[1]
# 2D histogram of (mean, difference): 0.05-dex bins in the mean, 0.04-dex bins in the difference
a.hist2d(mean, diff, bins=[np.arange(-3.0, 0.01, 0.05), np.arange(-1.5, 1.91, 0.04)], norm=LogNorm(), cmap="Blues", rasterized=True)
b = np.arange(-2.6, -0.49, 0.15)  # bins in the mean [Fe/H] [dex] for the running percentiles
# bin index of each star
i = np.digitize(mean, b)
# 16th, 50th, 84th percentiles of I - G in bins with at least 30 stars (NaN otherwise)
q = np.array([np.percentile(diff[i == k], [16, 50, 84]) if (i == k).sum() >= 30 else [np.nan] * 3 for k in range(1, len(b))])
# bin centres [dex]
xc = 0.5 * (b[1:] + b[:-1])
# running median (solid) and 16th/84th percentiles (dashed)
a.plot(xc, q[:, 1], color="k", lw=1)
a.plot(xc, q[:, 0], color="k", lw=0.6, ls="--")
a.plot(xc, q[:, 2], color="k", lw=0.6, ls="--")
# the adopted constant shift
a.axhline(tr["shift"], color=ORANGE, lw=1)
a.set(xlabel=r"([Fe/H]$_I$ + [Fe/H]$_G$)/2", ylabel=r"[Fe/H]$_I$ $-$ [Fe/H]$_G$", xlim=(-3.0, 0.0), ylim=(-1.5, 1.9))
# the Deming slope of I against G (from fig_rrab_feh_transform.py), above the panel
a.text(1, 1.03, f"Deming slope ${tr['deming_b']:.3f} \\pm {tr['deming_b_err']:.3f}$", transform=a.transAxes, ha="right")
a.text(0, 1.03, "(b)", transform=a.transAxes)

### Panel (c): against the spectroscopy

In [ ]:
a = ax[2]
# OGLE I (blue) and shifted Gaia G (orange) against the spectroscopic [Fe/H]
for k, col, lab in (("I", BLUE, "OGLE $I$"), ("G", ORANGE, f"Gaia $G + {tr['shift']:.2f}$")):
    # photometric minus spectroscopic [dex]
    d = s[k] - s.spec
    # legend label: median offset and robust sd (1.4826 x median absolute deviation) [dex]
    a.scatter(s.spec, s[k], s=7, color=col, edgecolor="white", lw=0.3,
              label=f"{lab}: {d.median():+.2f}, {1.4826 * (d - d.median()).abs().median():.2f}")
# dotted one-to-one line
x = np.array([-2.4, -0.4])
a.plot(x, x, color="0.4", lw=0.8, ls=":")
a.set(xlabel=r"[Fe/H]$_{\rm spec}$ (Gratton et al. 2004, C21 scale)", ylabel=r"[Fe/H]$_{\rm phot}$", xlim=(-2.4, -0.4), ylim=(-2.6, -0.2))
# legend above the panel; its title gives the number of matched stars
a.legend(loc="lower right", bbox_to_anchor=(1.0, 1.0), handletextpad=0.1, borderaxespad=0.1,
         title=f"{len(s)} stars: offset, $\\sigma$", title_fontsize=7, alignment="right")
a.text(0, 1.03, "(c)", transform=a.transAxes)

fig.savefig(ROOT / "figures" / "paper_rrab_feh.png")

In [ ]:
from IPython.display import Image
Image(filename='figures/paper_rrab_feh.png')

**Figure 3.2.** RRab photometric metallicities. (a) $I$-band [Fe/H] (OGLE light curves;
[Mullen et al. 2021](https://doi.org/10.3847/1538-4357/abefd4) relation as in Martínez-Vázquez et al. 2026) against $G$-band [Fe/H]
(*Gaia* light curves; Muraveva et al. 2025), in dex, for the stars with both (two-dimensional histogram, logarithmic colour scale);
dotted: one-to-one line; orange: the adopted constant shift of 0.22 dex. (b) Their difference against their
mean (dex), with the running median (solid black) and 16th and 84th percentiles (dashed black), the adopted shift (orange) and the
Deming slope (quoted above the panel). (c) Both photometric values against the spectroscopy of Gratton et al. (2004) on the
Crestani et al. (2021) scale, for the stars with all three: $I$ band (blue) and $G$ band plus the shift (orange); the legend gives
the median offset (photometric minus spectroscopic) and the robust scatter, in dex; dotted: one-to-one line.
`scripts/paper_rrab_feh.py`

### The shift between the two scales

`scripts/fig_rrab_feh_transform.py` measures, on the stars with both estimates and after dropping gross outliers of their
difference, the median difference between the $I$-band and $G$-band [Fe/H] and the Deming fit with its bootstrap errors. It writes
them to `results/rrab_feh_transform.csv`, from which the text takes the shift and the slope. Its figure shows the difference
against each estimate and against their mean, since binning on one noisy estimate alone produces a slope.

<details><summary>About <code>scripts/fig_rrab_feh_transform.py</code></summary>

```text
Is the OGLE I minus Gaia G (Muraveva+25) [Fe/H] difference a constant shift, or does it depend on [Fe/H]?

On the RRab with both estimates: the difference against I, G and their mean (binning on one noisy estimate alone produces a slope),
and a Deming fit I = a + b G with equal errors in both (the spectra give similar per-star errors, about 0.2-0.25 dex), bootstrapped.
Writes figures/rrab_feh_transform.png and results/rrab_feh_transform.csv.

Diagnostic (docs/figures.md: "RRab [Fe/H] scales"); not a paper figure. scripts/paper_text_data.py takes the shift and the Deming
slope of the text from results/rrab_feh_transform.csv.

Input: data/rrab.parquet (sources.yaml key release_rrab): feh_I (OGLE I band) and feh_G_m25 (Gaia G band, Muraveva+25) [dex].
Stars whose I - G difference lies more than 4 robust SD (1.4826 x MAD) from its median are dropped first.

Figure (1 x 4 panels, matplotlib constrained layout): (1)-(3) I minus G [dex] against I-band [Fe/H], G-band [Fe/H] and their
mean [dex]: running median (line) and 16th-84th percentiles (band) in 0.15-dex bins with >= 30 stars, dashed line the overall
median shift; (4) normalised histograms of I-band [Fe/H], G plus the median shift, and G transformed by the Deming fit [dex].

results/rrab_feh_transform.csv (one row): N, clipped (stars dropped), shift (median I - G) [dex], deming_a [dex], deming_b and
their bootstrap SDs (500 resamplings of the stars).

Run: .venv/bin/python scripts/fig_rrab_feh_transform.py
```
</details>

*Shown, not run here: data/rrab.parquet (scripts/build_rrab.py); a diagnostic whose figure is not on this site. The cells below are the script itself; run it from the repository with* `python scripts/fig_rrab_feh_transform.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

```python
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from lmcold.sources import DATA, ROOT
```

#### Deming regression
Slope and intercept of a straight line for equal errors in both variables.

```python
def deming(x, y):
    """Slope and intercept of y = a + b x for equal error variances in x and y.

    Parameters
    ----------
    x, y : numpy.ndarray
        Paired values with equal (unknown) error variances.

    Returns
    -------
    tuple of float
        (a, b): intercept (units of y) and slope. Note the order: intercept first.
    """
    # Population variances and covariance (ddof = 0).
    sxx, syy, sxy = np.var(x), np.var(y), np.cov(x, y, bias=True)[0, 1]
    # Deming slope for an error-variance ratio of 1 (orthogonal regression); the line passes through the means.
    b = (syy - sxx + np.sqrt((syy - sxx) ** 2 + 4 * sxy**2)) / (2 * sxy)
    return np.mean(y) - b * np.mean(x), b
```

#### Load the data
The RR Lyrae catalogue `data/rrab.parquet` (`scripts/build_rrab.py`).

```python
t = pd.read_parquet(DATA / "rrab.parquet")   # one row per RR Lyrae; feh_I, feh_G_m25 [dex]
```

#### The stars with both estimates
Stars with $I$-band and $G$-band [Fe/H]; gross outliers of the difference are
dropped.

```python
ov = t[t.feh_I.notna() & t.feh_G_m25.notna()]       # stars with both estimates
d = ov.feh_I - ov.feh_G_m25                          # I minus G [dex]
mad = 1.4826 * np.median(np.abs(d - d.median()))     # robust SD of the difference [dex]
ov = ov[np.abs(d - d.median()) < 4 * mad]  # drop gross outliers (wrong period or phi31) before the moment-based fit
```

#### Shift and Deming fit
The median difference and the Deming fit with its bootstrap errors, written to
`results/rrab_feh_transform.csv`.

```python
x, y = ov.feh_G_m25.to_numpy(), ov.feh_I.to_numpy()   # G-band and I-band [Fe/H] [dex], shape (N,)
a, b = deming(x, y)                                   # I = a + b G
# 500 bootstrap resamples of the stars (seed 2): shape (500, 2) of (a, b).
rng = np.random.default_rng(2)
boot = np.array([deming(x[i], y[i]) for i in (rng.integers(0, len(x), len(x)) for _ in range(500))])
shift = np.median(y - x)                              # median I - G [dex]
# One-row table: sample size, number clipped, shift, Deming a and b with bootstrap SDs.
res = pd.DataFrame([dict(N=len(x), clipped=len(t[t.feh_I.notna() & t.feh_G_m25.notna()]) - len(x), shift=shift,
                         deming_a=a, deming_a_err=boot[:, 0].std(), deming_b=b, deming_b_err=boot[:, 1].std())])
res.to_csv(ROOT / "results" / "rrab_feh_transform.csv", index=False, float_format="%.4f")
print(res.to_string(index=False, float_format="%.4f"))
```

#### The figure
The difference against each estimate and their mean, and the distributions after each transform.

```python
fig, ax = plt.subplots(1, 4, figsize=(17, 4.4), constrained_layout=True)
bins = np.arange(-2.6, -0.39, 0.15)   # 0.15-dex bins of [Fe/H], -2.6 to -0.5
for a_, (xv, xl) in zip(ax[:3], ((y, "[Fe/H] OGLE I"), (x, "[Fe/H] G, Muraveva+25"), ((x + y) / 2, "mean of the two"))):
    # Bin index of each star and bin centres; 16th, 50th, 84th percentiles of I - G in bins with >= 30 stars.
    i = np.digitize(xv, bins)
    xc = 0.5 * (bins[1:] + bins[:-1])
    q = np.array([np.percentile((y - x)[i == k], [16, 50, 84]) if (i == k).sum() >= 30 else [np.nan] * 3
                  for k in range(1, len(bins))])
    a_.fill_between(xc, q[:, 0], q[:, 2], color="#2a78d6", alpha=0.2, lw=0)
    a_.plot(xc, q[:, 1], color="#2a78d6", lw=2)
    a_.axhline(shift, color="0.3", lw=1, ls="--", label=f"median {shift:+.3f}")
    a_.set(xlabel=xl, ylabel="I minus G [dex]", ylim=(-0.8, 1.2), title=f"difference against {xl}")
ax[0].legend(frameon=False)
# Panel 4: normalised histograms (0.1-dex bins) of I, G + shift and G through the Deming line.
a_ = ax[3]
h = np.arange(-3.2, 0.21, 0.1)
a_.hist(y, h, histtype="step", lw=2, color="#2a78d6", density=True, label="OGLE I")
a_.hist(x + shift, h, histtype="step", lw=2, color="#eb6834", density=True, label=f"G {shift:+.2f}")
a_.hist(a + b * x, h, histtype="step", lw=2, color="#1baf7a", density=True, label=f"G Deming: {a:+.2f} + {b:.2f} G")
a_.set(xlabel="[Fe/H]", ylabel="density", title="I against transformed G (same stars)")
a_.legend(frameon=False, fontsize=8)
fig.savefig(ROOT / "figures" / "rrab_feh_transform.png", dpi=150)
```

## BOSS calibration

Figure 3.3 shows the 2 097 giants observed by both surveys that fix the two calibration
relations of the samples chapter, the velocity offset $v_{\rm BOSS}-v_{\rm APOGEE}=d_0+d_1t$ and
$[{\rm Fe/H}]_{\rm BOSS}=c_0+c_1[{\rm Fe/H}]_{\rm APOGEE}+c_2t+c_3s$, with $t$ the BOSS effective temperature and $s$ the inverse
BOSS signal-to-noise ratio, both scaled about a pivot as defined there. BOSSNet compresses the APOGEE scale by a factor of
$c_1$ = 0.73. The calibrated values follow APOGEE with no trend from [Fe/H] = -1.5
upwards and fall 0.15–0.17 dex low at -2 < [Fe/H] <
-1.5, the bias corrected in the stellar-mass chapter. The compression depends on S/N: fitted in bins of BOSS
S/N, the slope of BOSSNet against APOGEE [Fe/H] is 0.16 ± 0.23 (22 stars)
at 10–13, 0.51 ± 0.10 (41) at
13–16, 0.56 ± 0.07 (78) at
16–20, 0.67 ± 0.04 at
20–25 and 0.77–0.79 ± 0.02 above
30. Below S/N = 16 the calibrated values run 0.06 and
0.16 dex high, so we require S/N ≥ 16 for a good BOSS spectrum; above it the median offset
is within 0.02 dex in every S/N bin. Rebuilding the BOSS sample with S/N ≥ 20 instead
changes the total formed mass by -0.2 per cent, the mass below [Fe/H] = -1.5 by
-5 per cent and below -2 by +23 per cent, the scale lengths of the
metallicity-structure chapter by at most 0.28 kpc and the radial median [Fe/H] profile by at most
0.035 dex outside 1 kpc, all within the statistical errors. The velocity offset falls with
temperature as the relation for $v_{\rm BOSS}-v_{\rm APOGEE}$ describes.

<details><summary>About <code>scripts/paper_boss_calibration.py</code></summary>

```text
Paper figure for Sect. 2.1.2: the BOSS -> APOGEE calibration (Eqs 1, 2) on the stars observed by both surveys.

Run build_giants.py first (writes data/giants_overlap.parquet and results/boss_calibration.json).
(a) raw BOSSNet [Fe/H] against APOGEE [Fe/H], with the fitted relation at the pivot (t = s = 0) and the one-to-one line;
(b) calibrated BOSS minus APOGEE [Fe/H] against APOGEE [Fe/H]; (c) BOSS minus APOGEE velocity against BOSS Teff, with Eq. 1.
Lines and bands in (b, c): median and 16th-84th percentiles per bin. In-sample: all stars of the overlap enter the fit.

Inputs (both derived, not in data/sources.yaml; made by scripts/build_giants.py from data/spec_giants.parquet and
data/gaia_giants.parquet):
- data/giants_overlap.parquet: giants with good spectra in both APOGEE DR17 and BOSS DR20 (same Gaia DR3 source_id); columns
  source_id, teff_b [K], snr_b, v_b [km/s], feh_b (raw BOSSNet) from BOSS and v_a [km/s], feh_a from APOGEE (ASPCAP).
- results/boss_calibration.json, key 'calibration': coefficients d0, d1 (velocity, Eq. 1) and c0-c3 ([Fe/H], Eq. 2) of
  lmcold.giants.fit_calibration, fitted on all overlap stars.
Method: lmcold.giants.calibrate inverts Eq. 2 to put each BOSS [Fe/H] on the APOGEE scale; with t = (Teff_BOSS - 4200 K)/1000 K
and s = 20/(S/N)_BOSS - 1 the relation is [Fe/H]_BOSS = c0 + c1 [Fe/H]_APOGEE + c2 t + c3 s, and v_BOSS - v_APOGEE = d0 + d1 t.

Panels (1 x 3, equal square panels; points: individual overlap stars):
(a) x: APOGEE [Fe/H] [dex], y: raw BOSSNet [Fe/H] [dex], both -2.7 to 0.4; orange line Eq. 2 at t = s = 0 (c0 + c1 x),
    dashed one-to-one line.
(b) x: APOGEE [Fe/H] [dex], y: calibrated BOSS minus APOGEE [Fe/H] [dex], -1 to 1; running median and 16th-84th percentile
    band in 0.25-dex bins from -2.5 to 0.5 with >= 10 stars.
(c) x: BOSS Teff [K], 3600-5500, y: v_BOSS - v_APOGEE [km/s], -15 to 20; running median and band in 150-K bins from 3700 K;
    orange line Eq. 1 (d0 + d1 t).
Output: figures/paper_boss_calibration.png (appendix B, see docs/figures.md).
Run: .venv/bin/python scripts/paper_boss_calibration.py
```
</details>

**Data read by `scripts/paper_boss_calibration.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `data/giants_overlap.parquet` | Zenodo bundle 1 | Giants with both APOGEE and BOSS spectra (BOSS calibration) |
| `results/boss_calibration.json` | repository | calibration of the BOSS [Fe/H] and velocities to APOGEE on the stars observed by both; written by `python scripts/build_giants.py` |

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

In [ ]:
import json

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from lmcold import giants
from lmcold.layout import panel_grid
from lmcold.sources import DATA, ROOT

### Load the data
The stars observed by both surveys, `data/giants_overlap.parquet` (Zenodo bundle 1), and the fitted coefficients of
`results/boss_calibration.json` (both written by `scripts/build_giants.py`).

In [ ]:
# data/giants_overlap.parquet: stars with both spectra; BOSS (_b) and APOGEE (_a) [Fe/H], velocity, BOSS Teff and S/N
m = pd.read_parquet(DATA / 'giants_overlap.parquet')
# results/boss_calibration.json: coefficients c0, c1, ... (Eq. 2, [Fe/H]) and d0, d1 (Eq. 1, velocity) of the BOSS calibration
cal = json.load(open(ROOT / 'results/boss_calibration.json'))['calibration']

### The stars observed by both surveys
The BOSS [Fe/H] is put on the APOGEE scale.

In [ ]:
plt.style.use(ROOT / 'src/lmcold/paper.mplstyle')
# calibrate returns (velocity, [Fe/H]) on the APOGEE scale; only the calibrated [Fe/H] is kept (panel b).
_, m['feh_cal'] = giants.calibrate(cal, m.teff_b, m.snr_b, m.v_b, m.feh_b)
# Colours: blue for the running percentiles, orange for the fitted relations
blue, orange = '#2878b5', '#d75a20'

### Running percentiles
`running` draws the median and the 16th-84th percentile band of a quantity in bins, used in panels (b) and (c).

In [ ]:
def running(ax, x, y, edges, nmin=10):
    """Median and 16th-84th percentiles of y in bins of x with at least nmin stars.

    Draws the percentile band and the median line on `ax` at the bin centres; bins with fewer than `nmin` stars are skipped.

    Parameters
    ----------
    ax : matplotlib.axes.Axes
        Panel to draw on.
    x, y : numpy.ndarray
        Binned quantity and the quantity summarised (same units as the panel axes).
    edges : numpy.ndarray
        Bin edges in x.
    nmin : int, optional
        Minimum number of stars for a bin to be drawn.

    Returns
    -------
    None
    """
    i = np.digitize(x, edges) - 1  # bin index; -1 or len(edges)-1 for stars outside the edges, which no bin selects
    # Per bin with >= nmin stars: (bin centre, 16th, 50th, 84th percentiles of y)
    rows = [(0.5 * (edges[k] + edges[k + 1]), *np.percentile(y[i == k], [16, 50, 84])) for k in range(len(edges) - 1)
            if (i == k).sum() >= nmin]
    # Band between the 16th and 84th percentiles and the median line
    c, lo, med, hi = np.array(rows).T
    ax.fill_between(c, lo, hi, color=blue, alpha=.2, lw=0)
    ax.plot(c, med, '-', color=blue)

### The figure
(a) raw BOSSNet against APOGEE [Fe/H]; (b) residual after calibration; (c) velocity offset against BOSS $T_{\rm eff}$.

In [ ]:
# Three equal panels (sizes in inches); dot: style of the individual stars
fig, axes = panel_grid(1, 3, width=7, left=.5, right=.1, hgap=.55, bottom=.42, top=.3)
dot = dict(s=1.5, color='0.45', alpha=.5, lw=0, rasterized=True)
x = np.linspace(-2.7, .4, 2)  # end points of the straight lines in (a) [dex]

# (a) Raw BOSSNet against APOGEE [Fe/H]; Eq. 2 at the pivot (Teff = 4200 K, S/N = 20), whose slope c1 is the BOSSNet scale factor.
ax = axes[0, 0]
# Stars, one-to-one line and Eq. 2 at t = s = 0
ax.scatter(m.feh_a, m.feh_b, **dot)
ax.plot(x, x, '--', color='0.3', lw=.7, label='one to one')
ax.plot(x, cal['c0'] + cal['c1'] * x, color=orange, label=rf'Eq. 2, $t=s=0$: slope {cal["c1"]:.2f}')
ax.set(xlim=(-2.7, .4), ylim=(-2.7, .4), xlabel=r'[Fe/H]$_{\rm APOGEE}$', ylabel=r'[Fe/H]$_{\rm BOSS}$ (BOSSNet)',
       title=f'(a) {len(m):,} stars in both surveys')
ax.legend(loc='upper left')

# (b) Residual of the calibrated BOSS [Fe/H] against APOGEE [Fe/H].
ax = axes[0, 1]
# Residuals [dex] of each star and their running percentiles in 0.25-dex bins from -2.5
ax.scatter(m.feh_a, m.feh_cal - m.feh_a, **dot)
running(ax, m.feh_a.to_numpy(), (m.feh_cal - m.feh_a).to_numpy(), np.arange(-2.5, .5, .25))
ax.axhline(0, color='0.3', ls='--', lw=.7)
ax.set(xlim=(-2.7, .4), ylim=(-1, 1), xlabel=r'[Fe/H]$_{\rm APOGEE}$', ylabel='calibrated BOSS $-$ APOGEE [Fe/H] (dex)',
       title='(b) [Fe/H] after calibration')

# (c) Raw velocity difference against BOSS Teff, with the fitted linear trend (Eq. 1), d1 per 1000 K about T_PIVOT = 4200 K.
ax = axes[0, 2]
dv = (m.v_b - m.v_a).to_numpy()  # km/s
# Velocity differences and running percentiles in 150-K bins from 3700 K
ax.scatter(m.teff_b, dv, **dot)
running(ax, m.teff_b.to_numpy(), dv, np.arange(3700, 5600, 150))
t = np.linspace(3600, 5500, 2)  # K
# Eq. 1: v_BOSS - v_APOGEE = d0 + d1 (Teff - T_PIVOT) / 1000 K [km/s]
ax.plot(t, cal['d0'] + cal['d1'] * (t - giants.T_PIVOT) / 1000, color=orange, label='Eq. 1')
ax.axhline(0, color='0.3', ls='--', lw=.7)
ax.set(xlim=(3600, 5500), ylim=(-15, 20), xlabel=r'$T_{\rm eff,BOSS}$ (K)', ylabel=r'$v_{\rm BOSS}-v_{\rm APOGEE}$ (km s$^{-1}$)',
       title='(c) Velocity offset')
ax.legend(loc='upper right')
fig.savefig(ROOT / 'figures/paper_boss_calibration.png')
print('Saved paper_boss_calibration.png')

In [ ]:
from IPython.display import Image
Image(filename='figures/paper_boss_calibration.png')

**Figure 3.3.** BOSS on the APOGEE scale, from the 2 097 giants observed by both surveys
(grey points). (a) Raw BOSSNet against APOGEE [Fe/H] (dex), with the [Fe/H] relation at $t=s=0$ (orange; slope $c_1$ =
0.73) and the one-to-one line (dashed). (b) Calibrated BOSS minus APOGEE [Fe/H] (dex) against APOGEE [Fe/H].
(c) BOSS minus APOGEE velocity (km/s) against BOSS $T_{\rm eff}$ (K), with the velocity relation (orange). Blue lines and bands in
(b, c): median and 16th–84th percentiles per bin of at least 10 stars. All stars shown enter the fits; held-out values are given
in the samples chapter. `scripts/paper_boss_calibration.py`

## BOSSNet metallicities of hot giants

Figure 3.4 shows why the BOSS metallicities do not share out the mass above [Fe/H] = -1 (stellar-mass chapter).
Against APOGEE, for the stars observed by both with APOGEE [Fe/H] between -1 and -0.5, BOSS is unbiased below $T_{\rm eff}$ =
4300 K (offsets within 0.03 dex) but reads +0.05–0.06 dex high
at 4300–4500 K and +0.14 to +0.17 dex at
4500–4700 K, at both low and high S/N (panel a). At fixed magnitude and radius, the faint blue BOSS
giants (dereddened $G_{\rm BP}-G_{\rm RP}$ < 1.1) are 54 per cent metal-rich ([Fe/H] >
-0.5) at S/N < 25 and 12 per cent at S/N ≥ 25; in the
reddest colour bin (1.6–1.8) the shares fall to 7 and 4 per
cent, against 0–6 per cent for APOGEE at every colour
(panel b). The BOSS sample is dominated by hot, low-S/N giants, which the stars observed by both surveys barely contain (panel c),
so the calibration and the response cannot correct them; the stellar-mass chapter replaces the BOSS split above -1 with that of
APOGEE.

<details><summary>About <code>scripts/paper_boss_hot_bias.py</code></summary>

```text
Paper figure (appendix): BOSSNet over-estimates [Fe/H] of hot, low-S/N giants, which the calibration sample barely contains.

(a) Stars with both spectra (lmcold.mass_samples.overlap), APOGEE [Fe/H] -1 to -0.5: calibrated BOSS minus APOGEE [Fe/H] against
    BOSS Teff, for S/N < 25 and >= 25; medians in Teff bins with >= 8 stars.
(b) Weighted fraction of giants with [Fe/H] > -0.5 against dereddened BP-RP (lmcold.mass_samples.with_colour) at 16.25 <= G <= 17.5
    and 6 <= rho <= 12 deg (where APOGEE has faint giants; the metal-rich fraction falls with radius): BOSS (weights w x coverage x
    (1 - f_MW)) at S/N < 25 and >= 25, APOGEE (weights w).
(c) BOSS Teff distribution of the BOSS mass sample (weighted) and of the stars with both spectra, for S/N < 25 and >= 25.
Writes results/boss_hot_bias.json and figures/paper_boss_hot_bias.png.

Why: this bias is the reason the BOSS mass above [Fe/H] = -1 is shared among the reporting bins as the APOGEE giants of the same
ring x G x colour cell (lmcold.mass_samples.metal_rich_split, used by scripts/build_giant_mass.py).

Inputs (all derived; not in data/sources.yaml as files of their own):
- lmcold.mass_samples.samples(): BOSS and APOGEE mass samples (velocity-selected members with selection weights w) from
  data/boss_weights.parquet, data/apogee_weights.parquet, data/spec_giants.parquet and results/boss_calibration.json; BOSS
  [Fe/H] and v_los are on the APOGEE scale.
- data/giant_foreground.parquet (scripts/build_giant_mass.py): per BOSS giant the coverage fill factor `fill` (parent /
  covered parent, lmcold.selection.coverage_factor) and the Milky Way foreground fraction `f_mw` (lmcold.foreground).
- data/boss_parent.parquet (via with_colour): Gaia G, BP, RP and SFD E(B-V) for the dereddened colour c0 and rho.
Note: with_colour takes photometry from the BOSS parent, so APOGEE giants outside the BOSS parent drop out of (b).

Panels (1 x 3; legends above each panel):
(a) x: BOSS Teff [K], 3700-5300; y: calibrated BOSS - APOGEE [Fe/H] [dex], -0.5 to 0.6; points: overlap stars with APOGEE
    [Fe/H] in [-1, -0.5); lines: median per 200-K bin (>= 8 stars), orange S/N < 25, blue S/N >= 25.
(b) x: dereddened Gaia BP-RP [mag], 0.9-2.2; y: weighted fraction with [Fe/H] > -0.5, 0-0.85; BOSS S/N < 25, BOSS S/N >= 25,
    APOGEE; bins with >= 20 stars.
(c) x: BOSS Teff [K]; y: fraction per 200-K bin. Solid: BOSS mass sample weighted by W, normalised to the total weight of the
    sample; dashed: overlap stars, normalised to the number of overlap stars (so each pair of S/N curves sums to 1).
Output JSON results/boss_hot_bias.json: the bin edges, the curves of (a) and (b) (offset_*, n_overlap_*, frac_*, n_*), the
weight fraction of the BOSS sample that is hot (Teff > 4300 K) and low S/N, the same fraction among the overlap stars, and the
weighted fractions above [Fe/H] = -0.5 for BOSS (hot low-S/N, rest, all) and APOGEE.
Run: .venv/bin/python scripts/paper_boss_hot_bias.py   (about 2 s on a laptop, docs/run_times.txt)
```
</details>

**Data read by `scripts/paper_boss_hot_bias.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `data/apogee_weights.parquet` | Zenodo bundle 1 | APOGEE selection weights per candidate star (selection groups, coverage, w = N_cand / N_quality x pass factor in H x XP [M/H]) |
| `data/boss_parent.parquet` | Zenodo bundle 1 | Gaia DR3 stars within 12 deg of the LMC passing the rough cuts of mwm_magcloud_rgb_boss, with SFD E(B-V) and `eligible` |
| `data/boss_weights.parquet` | Zenodo bundle 1 | BOSS selection weights per eligible parent star (selection groups, w = N_elig / N_good x pass factor in G x XP [M/H] x ring) |
| `data/giant_foreground.parquet` | Zenodo bundle 1 | Milky Way foreground model per giant, from the mass analysis |
| `data/raw/skowron21/table1.dat.gz` | public archive | OGLE-IV red-clump E(V-I) map of the Magellanic Clouds, 81,684 boxes of 3.4-27.5 arcmin; Skowron et al. 2021, ApJS 252, 23 (2021ApJS..252...23S, doi:10.3847/1538-4365/abcb81) |
| `data/spec_giants.parquet` | Zenodo bundle 1 | APOGEE DR17 and SDSS-V DR20 BOSS spectra of LMC giant candidates before the Gaia match |
| `results/boss_calibration.json` | repository | calibration of the BOSS [Fe/H] and velocities to APOGEE on the stars observed by both; written by `python scripts/build_giants.py` |

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

In [ ]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lmcold.layout import panel_grid
from lmcold.mass_samples import samples, overlap, with_colour
from lmcold.sources import DATA, ROOT

SNR_SPLIT = 25.  # BOSS S/N separating the low- and high-S/N subsets (sample S/N >= lmcold.giants.BOSS_SNR_MIN = 16)
T_EDGES = np.arange(3700, 5301, 200)  # K, BOSS Teff bins of (a) and (c)
C_EDGES = np.array([.9, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.8, 2.2])  # mag, dereddened BP-RP bins of (b)

### Load the data
The BOSS and APOGEE mass samples through `lmcold.mass_samples.samples()` (built from the catalogues of Zenodo bundle 1), and the
coverage fill and foreground fraction of each BOSS giant from `data/giant_foreground.parquet` (written by
`scripts/build_giant_mass.py`).

In [ ]:
# lmcold.mass_samples.samples(): BOSS and APOGEE giants with calibrated [Fe/H], Teff, S/N and selection weight w
b, a, _ = samples()
# data/giant_foreground.parquet: per BOSS giant the coverage fill and the Milky Way foreground fraction f_mw
fg = pd.read_parquet(DATA / 'giant_foreground.parquet')

### The mass samples and the stars with both spectra
BOSS weights include the coverage fill and the foreground fraction; APOGEE uses its selection weights.

In [ ]:
# BOSS weight W = selection weight x coverage fill x LMC-member probability (1 - foreground fraction), as in the BOSS mass.
b = b.merge(fg[['source_id', 'fill', 'f_mw']], on='source_id', validate='one_to_one').assign(W=lambda t: t.w * t['fill'] * (1 - t.f_mw))
a = a.assign(W=a.w)  # APOGEE: selection weight only
# Overlap stars: feh_b (calibrated BOSS), feh_a (APOGEE), BOSS g and snr, plus BOSS Teff; d = BOSS - APOGEE [Fe/H] [dex].
o = overlap(b, a).merge(b[['source_id', 'teff']], on='source_id', validate='one_to_one')
o['d'] = o.feh_b - o.feh_a
b, a = with_colour(b), with_colour(a)  # adds g, c0 (dereddened BP-RP) and rho [deg]

### Binned statistics

In [ ]:
def binned(x, y, w, edges, f, nmin):
    """Statistic f(y, w) per bin of x; NaN for bins with fewer than nmin stars.

    Parameters
    ----------
    x : array-like
        Binned quantity (BOSS Teff [K] or dereddened BP-RP [mag]).
    y, w : numpy.ndarray
        Values ([Fe/H] or its BOSS - APOGEE difference [dex]) and weights per star.
    edges : numpy.ndarray
        Bin edges in x; stars outside them are ignored.
    f : callable
        f(y, w) -> float, evaluated on the stars of one bin.
    nmin : int
        Minimum number of stars (unweighted) in a bin.

    Returns
    -------
    numpy.ndarray
        One value per bin, shape (len(edges) - 1,).
    """
    # Used for the median offsets of panel (a) (f = mid) and the metal-rich fractions of panel (b) (f = frac).
    k_ = np.digitize(x, edges) - 1   # bin index of each star (-1 or len(edges) - 1: outside)
    # f on the stars of each bin with at least nmin stars, NaN otherwise.
    return np.array([f(y[k_ == i], w[k_ == i]) if (k_ == i).sum() >= nmin else np.nan for i in range(len(edges) - 1)])

### The curves and `results/boss_hot_bias.json`
Offsets against $T_{\rm eff}$, metal-rich fractions against colour, and the weight of hot, low-S/N giants.

In [ ]:
frac = lambda y, w: float(np.sum(w[y > -.5]) / np.sum(w))  # noqa: E731  weighted fraction with [Fe/H] > -0.5
mid = lambda y, w: float(np.median(y))  # noqa: E731  unweighted median (w ignored)
rich = o[(o.feh_a >= -1) & (o.feh_a < -.5)]  # overlap stars with APOGEE [Fe/H] in [-1, -0.5): panel (a)
# JSON output, started with the settings.
out = dict(snr_split=SNR_SPLIT, t_edges=T_EDGES.tolist(), c_edges=C_EDGES.tolist())
# Panel (b) subsamples: faint giants (16.25 <= G <= 17.5) at rho >= 6 deg (the parent ends at 12 deg).
faint_b, faint_a = (t[t.g.between(16.25, 17.5) & (t.rho >= 6)] for t in (b, a))
curves = {}   # name -> (bins,) array
# Panel (a): median offset (bins with >= 8 stars) and star counts per Teff bin, low and high S/N.
for lab, s in [('low', rich[rich.snr < SNR_SPLIT]), ('high', rich[rich.snr >= SNR_SPLIT])]:
    curves[f'offset_{lab}'] = binned(s.teff, s.d.to_numpy(), np.ones(len(s)), T_EDGES, mid, 8)
    curves[f'n_overlap_{lab}'] = np.histogram(s.teff, T_EDGES)[0]
# Panel (b): weighted fraction with [Fe/H] > -0.5 (bins with >= 20 stars) and star counts per colour bin.
for lab, s in [('boss_low', faint_b[faint_b.snr < SNR_SPLIT]), ('boss_high', faint_b[faint_b.snr >= SNR_SPLIT]), ('apogee', faint_a)]:
    curves[f'frac_{lab}'] = binned(s.c0, s.feh.to_numpy(), s.W.to_numpy(), C_EDGES, frac, 20)
    curves[f'n_{lab}'] = np.histogram(s.c0, C_EDGES)[0]
# "Hot, low-S/N" BOSS giants: Teff > 4300 K (above it the offset of panel a becomes positive) and S/N < 25.
hot = (b.teff > 4300) & (b.snr < SNR_SPLIT)
# Curves, the weight fraction of hot low-S/N BOSS giants, the same fraction among overlap stars (unweighted), and the
# metal-rich fractions of BOSS (hot low-S/N, the rest, all) and APOGEE.
out.update({k_: v.tolist() for k_, v in curves.items()},
           target_weight_hot_lowsnr=float(b.W[hot].sum() / b.W.sum()),
           overlap_fraction_hot_lowsnr=float(((o.teff > 4300) & (o.snr < SNR_SPLIT)).mean()),
           boss_frac_above_m05={'hot_lowsnr': frac(b.feh[hot].to_numpy(), b.W[hot].to_numpy()),
                                'rest': frac(b.feh[~hot].to_numpy(), b.W[~hot].to_numpy()), 'all': frac(b.feh.to_numpy(), b.W.to_numpy())},
           apogee_frac_above_m05=frac(a.feh.to_numpy(), a.W.to_numpy()))
(ROOT / 'results/boss_hot_bias.json').write_text(json.dumps(out, indent=1))
print(json.dumps(out, indent=1))

### The figure

In [ ]:
plt.style.use(ROOT / 'src/lmcold/paper.mplstyle')
fig, axes = panel_grid(1, 3, width=7, left=.55, right=.1, hgap=.6, bottom=.5, top=.6)
tm, cm = .5 * (T_EDGES[1:] + T_EDGES[:-1]), .5 * (C_EDGES[1:] + C_EDGES[:-1])  # bin centres [K], [mag]
# (a) BOSS - APOGEE [Fe/H] against BOSS Teff for the overlap stars at APOGEE -1..-0.5.
ax = axes[0, 0]
# Points: individual stars; line: median per Teff bin.
for lab, c, mk in [('low', '#d75a20', 'o'), ('high', '#2878b5', 's')]:
    s = rich[(rich.snr < SNR_SPLIT) if lab == 'low' else (rich.snr >= SNR_SPLIT)]
    ax.scatter(s.teff, s.d, s=2, color=c, alpha=.3, lw=0)
    ax.plot(tm, curves[f'offset_{lab}'], '-' + mk, ms=3, color=c,
            label=f"S/N {'<' if lab == 'low' else '≥'} {SNR_SPLIT:g} ({len(s)} stars)")
ax.axhline(0, color='.6', lw=.7)
ax.set(xlabel=r'BOSS $T_{\rm eff}$ [K]', ylabel='BOSS − APOGEE [Fe/H]', ylim=(-.5, .6), xlim=(3700, 5300),
       title='(a) APOGEE [Fe/H] −1 to −0.5')
# (b) Weighted metal-rich fraction against dereddened colour for faint outer giants.
ax = axes[0, 1]
# (curve key, colour, line and marker style, legend text) per sample.
for lab, c, ls, txt in [('boss_low', '#d75a20', '-o', f'BOSS, S/N < {SNR_SPLIT:g}'), ('boss_high', '#2878b5', '-s', f'BOSS, S/N ≥ {SNR_SPLIT:g}'),
                        ('apogee', 'k', '--^', 'APOGEE')]:
    ax.plot(cm, curves[f'frac_{lab}'], ls, ms=3, color=c, label=txt)
ax.set(xlabel=r'$(G_{\rm BP}-G_{\rm RP})_0$', ylabel='weighted fraction with [Fe/H] > −0.5', ylim=(0, .85),
       title=r'(b) $16.25\leq G\leq17.5$, $\rho\geq6^\circ$')
# (c) Teff distributions: BOSS mass sample (weighted) against the overlap stars that set the calibration.
ax = axes[0, 2]
for lab, c in [('low', '#d75a20'), ('high', '#2878b5')]:
    # S/N subset of the BOSS sample (m) and of the overlap stars (mo).
    m = (b.snr < SNR_SPLIT) if lab == 'low' else (b.snr >= SNR_SPLIT)
    mo = (o.snr < SNR_SPLIT) if lab == 'low' else (o.snr >= SNR_SPLIT)
    ax.hist(b.teff[m], T_EDGES, weights=b.W[m] / b.W.sum(), histtype='step', color=c, lw=1.3,
            label=f"BOSS sample, S/N {'<' if lab == 'low' else '≥'} {SNR_SPLIT:g}")
    ax.hist(o.teff[mo], T_EDGES, weights=np.full(mo.sum(), 1 / len(o)), histtype='step', color=c, lw=1, ls='--',
            label=f"both spectra, S/N {'<' if lab == 'low' else '≥'} {SNR_SPLIT:g}")
ax.set(xlabel=r'BOSS $T_{\rm eff}$ [K]', ylabel='fraction', xlim=(3700, 5300), title='(c) Who is calibrated')
# Legends above each panel.
for ax in axes[0]:
    ax.legend(loc='lower center', bbox_to_anchor=(.5, 1.12), ncol=1, frameon=False, fontsize=6.5)
fig.savefig(ROOT / 'figures/paper_boss_hot_bias.png', dpi=200)

In [ ]:
from IPython.display import Image
Image(filename='figures/paper_boss_hot_bias.png')

**Figure 3.4.** BOSSNet metallicities of hot giants. (a) Calibrated BOSS minus APOGEE [Fe/H] (dex) for the stars
observed by both surveys with APOGEE [Fe/H] between -1 and -0.5, against BOSS $T_{\rm eff}$ (K), for S/N < 25
(orange) and ≥ 25 (blue); points: stars; lines: medians in 200-K bins with at least
8 stars. (b) Weighted fraction of giants with [Fe/H] > -0.5 at 16.25 ≤ $G$ ≤
17.5 and $\rho$ ≥ 6°, against dereddened $G_{\rm BP}-G_{\rm RP}$ (mag): BOSS at S/N <
25 (orange circles) and ≥ 25 (blue squares), and APOGEE (black dashed, triangles).
(c) Distribution of BOSS $T_{\rm eff}$ (fraction per 200-K bin) of the weighted BOSS mass sample (solid) and of
the stars observed by both surveys (dashed), each for S/N < 25 (orange) and ≥ 25 (blue).
`scripts/paper_boss_hot_bias.py`